# Three check sources: check_master, PKG payments and the TM channel view (v1.0)

**PNC Treasury Management · DSI · Payment Knowledge Graph (PKG)**

**Question.** The rail reconciliation (`pkg_rail_recon` v1.1) found checks paid are the one real hole in PKG: about
half the dollars and a third of the count of the TM channel view, with 61% of customer-months holding no PKG check
at all. Another app reads checks from `cdihd01p_cdi.check_master`. Does that table hold the missing checks, and
which of the three sources should PKG use for checks?

| Source | What it is | Customer side |
|---|---|---|
| check_master | one row per check item processed (CPC), with drawer (`_from`) and depositor (`_to`) power ids | `cust_pwr_id_from` = the PNC customer who wrote the check (money out); `cust_pwr_id_to` = the PNC customer who deposited it (money in) |
| PKG | `neo4j_payments` / edges, category 7 | 7B check paid (out), 7C check deposited (in), 7A both sides PNC |
| channel view | `cdm_tm_cor_trans_agg_v`, account × month totals | `check_disbursement_amt` out; `check_dep_lockbox_amt`, `check_dep_remote_amt`, `branch_dep_amt` in |

The other app keeps only `dr_cr_flag = 'Debit'` rows and calls them incoming (only the depositor is a PNC customer),
outgoing (only the drawer) or customer to customer (both). That rule is one of the slices tested here, next to all
rows and the other flags.

| § | Question |
|---|---|
| 0 | check_master: columns, months, is `tran_id` one row per check, what kinds of rows it holds (flags, channels, types, sides) |
| 1 | The three sources per customer and month (cached work file) |
| 2 | Totals by month, in and out |
| 3 | Which pieces match: view against check_master slices, PKG against check_master, view against PKG |
| 4 | Item by item: are PKG's check payments the same items as check_master's (`trans_id` = `tran_id`)? |
| 5 | Where PKG falls short: does check_master fill the gap? |
| 6 | Answer sheet |

Read-only; writes a work file under `WORK_DIR` and csv files under `OUT_DIR`.

In [ ]:
# =============================================================================
# SETUP
# =============================================================================
import os, re, sys, time, math, warnings, threading, contextlib
import numpy as np
import pandas as pd

for _a, _t in [("object0", object), ("bool8", np.bool_), ("int0", np.intp),
               ("uint0", np.uintp), ("bool", np.bool_)]:
    if not hasattr(np, _a):
        setattr(np, _a, _t)
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
for _w in (ResourceWarning, DeprecationWarning, FutureWarning):
    warnings.filterwarnings("ignore", category=_w)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker

from pyspark.sql import SparkSession, functions as F, Window as W
from pyspark.sql.types import DateType, TimestampType, NumericType

DB = "bdahd01p_dlcdi1_cdi_tm"
CONFIG = dict(
    CM_TABLE    = "cdihd01p_cdi.check_master",
    CM_DATE     = "cpc_load_date",           # transaction date column ('YYYY-MM-DD'); the other app uses it as tran_date
    CM_AMT      = "cpc_tran_amt",
    CM_ID       = "tran_id",                 # 'CPC|YYYY-MM-DD|14 digits'; PKG 7B ids have the same form
    CM_FROM     = "cust_pwr_id_from",        # drawer is a PNC customer (money out)
    CM_TO       = "cust_pwr_id_to",          # depositor is a PNC customer (money in)
    CM_ATTRS    = ["dr_cr_flag", "on_us_flag", "channel_code_desc", "transaction_type_desc", "document_type", "fin_tranx"],
    # check_master slices compared per customer-month: (label, Spark SQL condition on the row)
    CM_SLICES = [
        ("all rows", "true"),
        ("Debit rows (other app's rule)", "dr_cr_flag = 'Debit'"),
        ("Credit rows", "dr_cr_flag = 'Credit'"),
        ("Debit, financial", "dr_cr_flag = 'Debit' AND fin_tranx = 'Financial'"),
        ("Debit, without returns", "dr_cr_flag = 'Debit' AND NOT (lower(transaction_type_desc) LIKE '%return%')"),
    ],
    EDGE_TABLE  = f"{DB}.pkg_edge_monthly",
    PAY_TABLE   = "dsihd01p_dsi.neo4j_payments",
    DIM_TABLE   = f"{DB}.pkg_cust_dim",
    CHAN_TABLE  = "clhhd01p_clh.cdm_tm_cor_trans_agg_v",
    CHAN_ID     = "cust_pwr_id", CHAN_KEY = "time_key",
    VIEW_OUT    = ["check_disbursement_amt"],
    VIEW_IN     = ["check_dep_lockbox_amt", "check_dep_remote_amt", "branch_dep_amt"],
    FIRST_MONTH = "2024-01", LAST_MONTH = None,   # None = last month in all three sources
    LINK_MONTHS = 2,           # §4 item-by-item link on the last N months of the window (reads neo4j_payments)
    PROFILE_TOP = 40,          # §0 kinds of rows shown (all in cm_profile.csv)
    TOL = (0.01, 0.05),
    ID = "customer_pwr_id",
    WORK_DIR = "/user/pk36814/pkg_work/check_recon",
    REBUILD = False,
    OUT_DIR = "../eda/check_recon",
    SHUFFLE_PARTITIONS = 400,
)
ID = CONFIG["ID"]

spark = (SparkSession.builder.appName("pkg_check_recon")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "4g")
         .config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g")
         .config("spark.executor.memoryOverhead", "1g")
         .config("spark.yarn.am.memory", "4g")
         .config("spark.yarn.am.memoryOverhead", "1g")
         .config("spark.dynamicAllocation.maxExecutors", "512")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.kryoserializer.buffer.max", "128m")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.shuffle.partitions", str(CONFIG["SHUFFLE_PARTITIONS"]))
         .enableHiveSupport().getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
try:
    _L = spark._jvm.org.apache.logging.log4j
    for _n in ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage",
               "org.apache.kafka", "org.apache.spark.sql.catalyst.util.package",
               "org.apache.spark.network", "io.netty"):
        _L.core.config.Configurator.setLevel(_n, _L.Level.OFF)
except Exception:
    pass
spark.conf.set("spark.sql.shuffle.partitions", str(CONFIG["SHUFFLE_PARTITIONS"]))

try:
    from IPython.display import display, HTML
    _NB = "ipykernel" in sys.modules
except Exception:
    _NB = False

def _fmt(v):
    if v is None or (isinstance(v, float) and (math.isnan(v) or math.isinf(v))):
        return ""
    if isinstance(v, (bool, np.bool_)):
        return "yes" if v else "no"
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        a = abs(float(v))
        if a >= 1000:
            return f"{v:,.0f}"
        if a >= 10:
            return f"{v:,.1f}"
        return f"{v:,.3f}".rstrip("0").rstrip(".") if a else "0"
    return str(v)


def _html(df):
    d = df.copy()
    for c in d.columns:
        d[c] = d[c].map(_fmt)
    return d.to_html(index=False, border=0, escape=True)


class _Cell:
    """Progress lines and result tables of one cell, rendered as a single output."""
    def __init__(self):
        self.lines, self.tables, self.h = [], [], None

    def start(self, title):
        self.title, self.lines, self.tables = title, [], []
        self.h = display(HTML(""), display_id=True) if _NB else None
        self._draw()

    def log(self, txt, replace_last=False):
        if replace_last and self.lines:
            self.lines[-1] = txt
        else:
            self.lines.append(txt)
        self._draw()
        if not _NB:
            print("   ", txt, flush=True)

    def show(self, *blocks):
        self.tables = list(blocks)
        self._draw()
        if not _NB:
            for lab, df in blocks:
                print(f"\n== {lab}")
                print(df.to_string(index=False, max_rows=80))

    def _draw(self):
        if self.h is None:
            return
        css = ("<style>.pkgc table{border-collapse:collapse;font-size:12px;margin:2px 0 12px}"
               ".pkgc th{background:#f1f0ec;color:#0b0b0b;text-align:left;padding:3px 8px;border-bottom:1px solid #b8b7b0}"
               ".pkgc td{padding:2px 8px;border-bottom:1px solid #e6e5e0}"
               ".pkgc h4{margin:10px 0 3px;font-size:13px}.pkgc .pg{color:#8a8985;font-size:12px}</style>")
        body = f"<div class='pkgc'><h3 style='margin:4px 0'>{self.title}</h3>"
        body += "".join(f"<div class='pg'>{x}</div>" for x in self.lines)
        body += "".join(f"<h4>{lab}</h4>{_html(df)}" for lab, df in self.tables)
        self.h.update(HTML(css + body + "</div>"))


CELL = _Cell()
TIMINGS = []


@contextlib.contextmanager
def step(name):
    """Timed step with a live line: elapsed minutes and Spark tasks of the running stage."""
    t0 = time.time()
    CELL.log(f"running: {name} (started {time.strftime('%H:%M')})")
    stop = threading.Event()

    def _tick():
        while not stop.wait(30):
            msg = f"running: {name} ({(time.time() - t0) / 60:.1f} min"
            try:
                st = spark.sparkContext.statusTracker()
                ids = st.getActiveStageIds()
                if ids:
                    info = st.getStageInfo(max(ids))
                    msg += f", stage {info.stageId}: {info.numCompletedTasks:,}/{info.numTasks:,} tasks"
            except Exception:
                pass
            if _NB:
                CELL.log(msg + ")", replace_last=True)

    th = threading.Thread(target=_tick, daemon=True)
    th.start()
    try:
        yield
    finally:
        stop.set()
        mins = (time.time() - t0) / 60
        TIMINGS.append((name, round(mins, 2)))
        if _NB:
            CELL.log(f"done: {name} ({mins:.1f} min)", replace_last=True)
        else:
            print(f"    done: {name} ({mins:.1f} min)", flush=True)


GATES = []


def gate(name, ok, detail, action=""):
    GATES.append(dict(gate=name, result="PASS" if ok else "FAIL", detail=detail, action=action))


def gates_frame():
    return pd.DataFrame(GATES, columns=["gate", "result", "detail", "action"])



OUT_DIR = CONFIG["OUT_DIR"]
os.makedirs(OUT_DIR, exist_ok=True)
FIG_INDEX, ANSWERS = [], []
INK, INK2, GRID, SURF = "#0b0b0b", "#52514e", "#e6e5e0", "#fcfcfb"
C1, C2, C3, C4 = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
plt.rcParams.update({
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "axes.edgecolor": "#b8b7b0", "axes.labelcolor": INK2, "text.color": INK,
    "xtick.color": INK2, "ytick.color": INK2, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.8, "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "figure.dpi": 110,
})


def savefig(fig, name, caption):
    fig.savefig(os.path.join(OUT_DIR, f"{name}.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    FIG_INDEX.append(dict(file=f"{name}.png", caption=caption))


def ans(q, a):
    ANSWERS.append(dict(question=q, answer=a))


def table_months(tbl, col="month"):
    try:
        p = spark.sql(f"SHOW PARTITIONS {tbl}").toPandas().pipe(_nodec).iloc[:, 0]
        m = sorted({x.split(f"{col}=")[1].split("/")[0] for x in p if f"{col}=" in x})
        if m:
            return m
    except Exception:
        pass
    return sorted(str(r[0]) for r in spark.table(tbl).select(col).distinct().collect())


def mi_col(c):
    c = F.col(c) if isinstance(c, str) else c
    return (F.substring(c, 1, 4).cast("int") * 12 + F.substring(c, 6, 2).cast("int")).cast("int")


def mi2m(i):
    i = int(i)
    return f"{(i - 1) // 12:04d}-{(i - 1) % 12 + 1:02d}"


def m2mi(m):
    return int(m[:4]) * 12 + int(m[5:7])


def _nodec(pdf):
    """Spark DECIMAL columns arrive as Python Decimal objects; turn them into floats."""
    import decimal
    for c in pdf.columns:
        if pdf[c].dtype == object and pdf[c].map(lambda v: isinstance(v, decimal.Decimal)).any():
            pdf[c] = pdf[c].map(lambda v: float(v) if isinstance(v, decimal.Decimal) else v).astype(float)
    return pdf


def pct_tbl(df, cols):
    df = df.copy()
    for c in cols:
        df[c] = df[c].map(lambda v: "" if pd.isna(v) else f"{100 * v:.1f}%")
    return df


BANDS = [(-np.inf, 1_000, "under 1K"), (1_000, 10_000, "1K-10K"), (10_000, 100_000, "10K-100K"),
         (100_000, 1_000_000, "100K-1M"), (1_000_000, np.inf, "1M and over")]


def band(col):
    e = F.lit(BANDS[-1][2])
    for lo, hi, lab in reversed(BANDS[:-1]):
        e = F.when(F.col(col) < hi, F.lit(lab)).otherwise(e)
    return e


import hashlib, json, itertools


def pid_col(c, dtype):
    """power id as text without leading zeros; blank, '-1' and '0' = missing"""
    x = F.col(c)
    if dtype not in ("string",) and not dtype.startswith("varchar") and not dtype.startswith("char"):
        x = x.cast("decimal(38,0)")
    s = F.regexp_replace(F.trim(x.cast("string")), r"^0+", "")
    return F.when(s.isNull() | (s == "") | (s == "-1"), F.lit(None)).otherwise(s)


def months_between(a, b):
    return [mi2m(i) for i in range(m2mi(a), m2mi(b) + 1)]


_pf = lambda v: "n/a" if v is None or pd.isna(v) else f"{v:.1%}"
print("setup done")

## §0 · check_master: what it holds

One pass over check_master from `FIRST_MONTH`: rows and dollars by month, by which side is a PNC customer (drawer
only, depositor only, both, neither) and by the row attributes (`CM_ATTRS`). The window is every month all three
sources hold. Then: is `tran_id` one row per check, or does a check carry several rows (debit and credit legs)?

In [ ]:
CELL.start("§0 · check_master")
with step("schemas and months of the view and the edges"):
    cm = spark.table(CONFIG["CM_TABLE"])
    CMT = dict(cm.dtypes)
    _need = [CONFIG["CM_DATE"], CONFIG["CM_AMT"], CONFIG["CM_ID"], CONFIG["CM_FROM"], CONFIG["CM_TO"]]
    _miss = [c for c in _need if c not in CMT]
    if _miss:
        raise ValueError(f"{CONFIG['CM_TABLE']} has no column {_miss}")
    ATTRS = [c for c in CONFIG["CM_ATTRS"] if c in CMT]
    _attr_missing = [c for c in CONFIG["CM_ATTRS"] if c not in CMT]
    dim = spark.table(CONFIG["DIM_TABLE"])
    chv = spark.table(CONFIG["CHAN_TABLE"])
    CHT = dict(chv.dtypes)
    K = CONFIG["CHAN_KEY"]
    _kexpr = (F.date_format(F.col(K), "yyyyMMdd") if CHT[K] in ("date", "timestamp")
              else F.regexp_replace(F.col(K).cast("string"), "-", ""))
    _keys = chv.select(_kexpr.alias("key")).groupBy("key").count().toPandas()
    _keys["key"] = _keys.key.astype(str).str.replace(r"\.0$", "", regex=True)
    _keys["month"] = np.where(_keys.key.str.len().isin([6, 8]), _keys.key.str[:4] + "-" + _keys.key.str[4:6], None)
    _keys = _keys[_keys.month.notna()]
    _kpm = _keys.groupby("month").agg(keys=("key", "nunique"), last_key=("key", "max")).reset_index()
    DAILY_KEYS = bool(len(_kpm) and _kpm["keys"].median() >= 15)
    edge_months = table_months(CONFIG["EDGE_TABLE"])
    _date = F.substring(F.col(CONFIG["CM_DATE"]).cast("string"), 1, 10)
    _dtt = CMT[CONFIG["CM_DATE"]]

    def _dr(lo, hi=None):
        """date range on the raw column, so a partition on it is pruned"""
        c = F.col(CONFIG["CM_DATE"])
        if _dtt in ("date", "timestamp"):
            out = c >= F.lit(lo).cast("date")
            return out & (c < F.date_add(F.lit(hi).cast("date"), 1)) if hi else out
        out = c >= F.lit(lo)
        return out & (c <= F.lit(hi + "~")) if hi else out
    FROMP = pid_col(CONFIG["CM_FROM"], CMT[CONFIG["CM_FROM"]])
    TOP = pid_col(CONFIG["CM_TO"], CMT[CONFIG["CM_TO"]])
    SIDE = (F.when(FROMP.isNotNull() & TOP.isNotNull(), "both PNC").when(FROMP.isNotNull(), "drawer only (out)")
            .when(TOP.isNotNull(), "depositor only (in)").otherwise("neither"))
    _A = [F.coalesce(F.col(c).cast("string"), F.lit("(null)")).alias(c) for c in ATTRS]
with step("check_master rows and dollars by month, side and kind (one pass)"):
    PROF = (cm.where(_dr(f"{CONFIG['FIRST_MONTH']}-01"))
            .select(F.substring(_date, 1, 7).alias("month"), SIDE.alias("side"), *_A,
                    F.abs(F.col(CONFIG["CM_AMT"]).cast("double")).alias("amt"))
            .groupBy("month", "side", *ATTRS).agg(F.count(F.lit(1)).alias("rows"), F.sum("amt").alias("dollars"))
            .toPandas().pipe(_nodec))
cm_months = sorted(PROF.month.dropna().unique())
last = CONFIG["LAST_MONTH"] or min(max(cm_months), max(edge_months), max(_kpm.month))
MONTHS = [m for m in months_between(CONFIG["FIRST_MONTH"], last) if m in cm_months and m in edge_months and m in set(_kpm.month)]
if not MONTHS:
    raise ValueError("No month is in all three sources.")
USE_KEYS = (_keys[_keys.month.isin(MONTHS)].key.tolist() if DAILY_KEYS else _kpm[_kpm.month.isin(MONTHS)].last_key.tolist())
PW = PROF[PROF.month.isin(MONTHS)]
_bym = PW.groupby("month")[["rows", "dollars"]].sum().reset_index()
_bys = PW.groupby("side")[["rows", "dollars"]].sum().reset_index()
_bys["dollar_share"] = _bys.dollars / _bys.dollars.sum()
KINDS = PW.groupby(["side"] + ATTRS)[["rows", "dollars"]].sum().reset_index().sort_values("dollars", ascending=False)
KINDS["dollar_share"] = KINDS.dollars / KINDS.dollars.sum()
KINDS.to_csv(os.path.join(OUT_DIR, "cm_profile.csv"), index=False)
_one = {}
for c in ATTRS:
    t = PW.groupby(c)[["rows", "dollars"]].sum().reset_index().sort_values("dollars", ascending=False)
    t["dollar_share"] = t.dollars / t.dollars.sum()
    _one[c] = t.head(12)
with step("is tran_id one row per check? (last month of the window)"):
    _lm = MONTHS[-1]
    _u = (cm.where(_dr(f"{_lm}-01", f"{_lm}-31"))
          .groupBy(CONFIG["CM_ID"]).agg(F.count(F.lit(1)).alias("r"),
                                         *([F.countDistinct("dr_cr_flag").alias("flags")] if "dr_cr_flag" in CMT else []),
                                         F.countDistinct(F.col(CONFIG["CM_AMT"])).alias("amts"))
          .groupBy("r").agg(F.count(F.lit(1)).alias("ids"),
                            *([F.sum(F.when(F.col("flags") > 1, 1).otherwise(0)).alias("ids_with_both_flags")] if "dr_cr_flag" in CMT else []),
                            F.sum(F.when(F.col("amts") > 1, 1).otherwise(0)).alias("ids_with_different_amounts"))
          .orderBy("r").toPandas())
    _u = _u.rename(columns={"r": "rows_per_tran_id"})
    ONE_ROW = float(_u.loc[_u.rows_per_tran_id == 1, "ids"].sum() / max(_u.ids.sum(), 1))
_win = pd.DataFrame([dict(item="window (all three sources)", value=f"{MONTHS[0]} .. {MONTHS[-1]} ({len(MONTHS)} months)"),
                     dict(item="check_master months from FIRST_MONTH", value=f"{cm_months[0]} .. {cm_months[-1]}"),
                     dict(item="attribute columns used", value=", ".join(ATTRS)),
                     dict(item="attribute columns not found", value=", ".join(_attr_missing) or "none"),
                     dict(item="view period key", value="daily rows, summed" if DAILY_KEYS else "snapshots, last key of each month")])
CELL.show(("Window", _win), ("Rows and dollars by month (window)", _bym),
          ("By which side is a PNC customer", pct_tbl(_bys, ["dollar_share"])),
          *[(f"By {c}", pct_tbl(t, ["dollar_share"])) for c, t in _one.items()],
          (f"Kinds of rows, top {CONFIG['PROFILE_TOP']} by dollars (all in cm_profile.csv)",
           pct_tbl(KINDS.head(CONFIG["PROFILE_TOP"]), ["dollar_share"])),
          (f"Rows per {CONFIG['CM_ID']} ({_lm})", _u))
ans("Window", f"{MONTHS[0]} .. {MONTHS[-1]} ({len(MONTHS)} months)")
ans(f"Share of {CONFIG['CM_ID']} with one row ({_lm})", f"{ONE_ROW:.1%}")
ans("check_master dollars by side (drawer only / depositor only / both / neither)",
    " / ".join(_pf(_bys.set_index("side").dollar_share.get(s)) for s in ["drawer only (out)", "depositor only (in)", "both PNC", "neither"]))

## §1 · The three sources per customer and month

- **check_master:** a row counts as money out for the drawer's power id and money in for the depositor's (a check
  between two PNC customers counts for both, as in PKG). Each slice in `CM_SLICES` is summed separately;
  `both_pnc` marks rows where both sides are PNC customers.
- **PKG:** category 7 from the edges (5C/6C duplicates and debit card already out), read from each customer's side,
  mdm id mapped to power id: 7B out (outside payee), 7C in (outside drawer), 7A both sides PNC.
- **View:** check columns (`VIEW_OUT`, `VIEW_IN`) and their units, accounts summed per power id.

In [ ]:
CELL.start("§1 · Three sources per customer and month")
SIG = hashlib.md5(json.dumps({**{k: CONFIG[k] for k in ["CM_TABLE", "CM_DATE", "CM_AMT", "CM_FROM", "CM_TO", "EDGE_TABLE", "DIM_TABLE",
                                                        "CHAN_TABLE", "CHAN_ID", "CHAN_KEY", "VIEW_OUT", "VIEW_IN"]},
                              "attrs": ATTRS, "months": MONTHS}, default=str).encode()).hexdigest()[:8]


def cached(fn, name):
    path = f"{CONFIG['WORK_DIR']}/sig={SIG}/{name}"
    if not CONFIG["REBUILD"]:
        try:
            out = spark.read.parquet(path)
            out.limit(1).count()
            CELL.log(f"reused work file: {name}")
            return out
        except Exception:
            pass
    fn().write.mode("overwrite").parquet(path)
    return spark.read.parquet(path)


_dt = dict(dim.dtypes)
MAP = (dim.select(F.col("mdm_id").cast("string").alias("mdm"), pid_col(ID, _dt[ID]).alias("pid"))
       .where(F.col("pid").isNotNull()).dropDuplicates(["mdm"]))
NAME = (dim.select(pid_col(ID, _dt[ID]).alias("pid"), "customer_name").where(F.col("pid").isNotNull())
        .groupBy("pid").agg(F.min("customer_name").alias("customer_name"))) if "customer_name" in dim.columns else None


def _cm_long():
    base = (cm.where(_dr(f"{MONTHS[0]}-01", f"{MONTHS[-1]}-31"))
            .select(F.substring(_date, 1, 7).alias("month"), FROMP.alias("pfrom"), TOP.alias("pto"), *_A,
                    F.abs(F.col(CONFIG["CM_AMT"]).cast("double")).alias("amt")))
    base = base.withColumn("both_pnc", F.col("pfrom").isNotNull() & F.col("pto").isNotNull())
    o = base.where(F.col("pfrom").isNotNull()).select(F.col("pfrom").alias("pid"), "month", F.lit("out").alias("dir"), "both_pnc", *ATTRS, "amt")
    i = base.where(F.col("pto").isNotNull()).select(F.col("pto").alias("pid"), "month", F.lit("in").alias("dir"), "both_pnc", *ATTRS, "amt")
    return (o.unionByName(i).groupBy("pid", "month", "dir", "both_pnc", *ATTRS)
            .agg(F.sum("amt").alias("a"), F.count(F.lit(1)).alias("n")))


def _pkg():
    e = (spark.table(CONFIG["EDGE_TABLE"]).where(F.col("month").isin(MONTHS))
         .withColumn("code", F.upper(F.regexp_extract(F.col("category").cast("string"), r"^\s*(\d[A-Ca-c]\d?)", 1)))
         .where(F.col("code").startswith("7"))
         .select("month", "src_id", "src_kind", "dst_id", "dst_kind", "code", F.col("is_self").cast("boolean").alias("is_self"),
                 F.col("amount").cast("double").alias("amt"), F.col("n_txn").cast("long").alias("n")))
    o = e.where(F.col("src_kind") == "CUST").select("month", F.col("src_id").cast("string").alias("mdm"), F.col("dst_kind").alias("okind"),
                                                    F.col("dst_id").cast("string").alias("other"), F.lit("out").alias("dir"), "is_self", "amt", "n")
    i = e.where(F.col("dst_kind") == "CUST").select("month", F.col("dst_id").cast("string").alias("mdm"), F.col("src_kind").alias("okind"),
                                                    F.col("src_id").cast("string").alias("other"), F.lit("in").alias("dir"), "is_self", "amt", "n")
    r = o.unionByName(i).join(MAP, "mdm").join(MAP.select(F.col("mdm").alias("other"), F.col("pid").alias("opid")), "other", "left")
    r = r.withColumn("ext", F.col("okind") != "CUST")
    return (r.groupBy("pid", "month").agg(
        *[F.sum(F.when(F.col("dir") == d, F.col(x)).otherwise(0)).alias(f"pkg_{d}_{y}") for d in ("out", "in") for x, y in (("amt", "a"), ("n", "n"))],
        *[F.sum(F.when((F.col("dir") == d) & F.col("ext"), F.col(x)).otherwise(0)).alias(f"pkgx_{d}_{y}")
          for d in ("out", "in") for x, y in (("amt", "a"), ("n", "n"))]))


_vu = lambda c: next((c[:-4] + s for s in ("_units", "_unit", "_cnt", "_count") if c.endswith("_amt") and c[:-4] + s in CHT), None)
VCOLS = [c for c in CONFIG["VIEW_OUT"] + CONFIG["VIEW_IN"] if c in CHT]
UNIT_OF = {c: _vu(c) for c in VCOLS if _vu(c)}


def _view():
    v = chv.where(_kexpr.isin(USE_KEYS))
    _m = F.concat(F.substring(_kexpr, 1, 4), F.lit("-"), F.substring(_kexpr, 5, 2))
    cols = VCOLS + list(UNIT_OF.values())
    return (v.select(pid_col(CONFIG["CHAN_ID"], CHT[CONFIG["CHAN_ID"]]).alias("pid"), _m.alias("month"),
                     *[F.coalesce(F.col(c).cast("double"), F.lit(0.0)).alias(c) for c in cols])
            .where(F.col("pid").isNotNull()).groupBy("pid", "month").agg(*[F.sum(c).alias(c) for c in cols]))


SL = CONFIG["CM_SLICES"] + [("Debit, other side outside PNC", "dr_cr_flag = 'Debit' AND NOT both_pnc")]
SL = [(lab, cond) for lab, cond in SL if all(a in ATTRS for a in CONFIG["CM_ATTRS"] if re.search(rf"\b{a}\b", cond))]
with step("check_master per customer, month and kind"):
    CML = cached(_cm_long, "cm_long")
with step("PKG checks per customer and month"):
    PKG = cached(_pkg, "pkg")
with step("view check columns per customer and month"):
    VIEW = cached(_view, "view")
with step("one row per customer and month"):
    def _join():
        aggs = []
        for i, (lab, cond) in enumerate(SL):
            for d in ("out", "in"):
                c = F.expr(cond) & (F.col("dir") == d)
                aggs += [F.sum(F.when(c, F.col("a")).otherwise(0.0)).alias(f"cm{i}_{d}_a"),
                         F.sum(F.when(c, F.col("n")).otherwise(0)).alias(f"cm{i}_{d}_n")]
        cmw = CML.groupBy("pid", "month").agg(*aggs).withColumn("in_cm", F.lit(1))
        j = (VIEW.withColumn("in_view", F.lit(1)).join(PKG.withColumn("in_pkg", F.lit(1)), ["pid", "month"], "full")
             .join(cmw, ["pid", "month"], "full"))
        return j.na.fill(0, [c for c in j.columns if c not in ("pid", "month")])
    J = cached(_join, "joined").persist()
    NROW = J.count()
_sl = pd.DataFrame([dict(slice=i, label=lab, condition=cond) for i, (lab, cond) in enumerate(SL)])
CELL.show(("Rows", pd.DataFrame([dict(customer_months=NROW, work_file=f"{CONFIG['WORK_DIR']}/sig={SIG}")])),
          ("check_master slices compared", _sl))

## §2 · Totals by month

Dollars per source and direction over all customer-months (any source), then as a share of the view for customers
in the view. check_master is shown for each slice; the "Debit" slice is the other app's rule.

In [ ]:
CELL.start("§2 · Totals")
VOUT = sum((F.col(c) for c in CONFIG["VIEW_OUT"] if c in CHT), F.lit(0.0))
VIN = sum((F.col(c) for c in CONFIG["VIEW_IN"] if c in CHT), F.lit(0.0))
VOUT_N = sum((F.col(UNIT_OF[c]) for c in CONFIG["VIEW_OUT"] if c in UNIT_OF), F.lit(0.0))
VIN_N = sum((F.col(UNIT_OF[c]) for c in CONFIG["VIEW_IN"] if c in UNIT_OF), F.lit(0.0))
with step("totals"):
    aggs = [F.sum(VOUT).alias("view_out"), F.sum(VIN).alias("view_in"), F.sum(VOUT_N).alias("view_out_n"), F.sum(VIN_N).alias("view_in_n")]
    for d in ("out", "in"):
        aggs += [F.sum(f"pkg_{d}_a").alias(f"pkg_{d}"), F.sum(f"pkg_{d}_n").alias(f"pkg_{d}_n")]
        for i in range(len(SL)):
            aggs += [F.sum(f"cm{i}_{d}_a").alias(f"cm{i}_{d}"), F.sum(f"cm{i}_{d}_n").alias(f"cm{i}_{d}_n")]
    TALL = J.agg(*aggs).toPandas().pipe(_nodec).iloc[0]
    TV = J.where(F.col("in_view") == 1).agg(*aggs).toPandas().pipe(_nodec).iloc[0]
    MON = J.where(F.col("in_view") == 1).groupBy("month").agg(*aggs).orderBy("month").toPandas().pipe(_nodec)
TOT = []
for d in ("out", "in"):
    rows = [("channel view", f"view_{d}"), ("PKG", f"pkg_{d}")] + [(f"check_master: {lab}", f"cm{i}_{d}") for i, (lab, _) in enumerate(SL)]
    for lab, k in rows:
        TOT.append(dict(direction=d, source=lab, dollars_all=TALL[k], items_all=TALL[f"{k}_n"], dollars_view_customers=TV[k],
                        items_view_customers=TV[f"{k}_n"], dollar_share_of_view=TV[k] / TV[f"view_{d}"] if TV[f"view_{d}"] else np.nan,
                        item_share_of_view=TV[f"{k}_n"] / TV[f"view_{d}_n"] if TV[f"view_{d}_n"] else np.nan))
TOT = pd.DataFrame(TOT)
SHM = MON[["month"]].copy()
for d in ("out", "in"):
    SHM[f"PKG {d}"] = MON[f"pkg_{d}"] / MON[f"view_{d}"].replace(0, np.nan)
    for i, (lab, _) in enumerate(SL):
        SHM[f"cm{i} {d}"] = MON[f"cm{i}_{d}"] / MON[f"view_{d}"].replace(0, np.nan)
TOT.to_csv(os.path.join(OUT_DIR, "totals.csv"), index=False)
SHM.to_csv(os.path.join(OUT_DIR, "share_by_month.csv"), index=False)
fig, axs = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, d in zip(axs, ("out", "in")):
    ax.plot(pd.to_datetime(SHM.month), SHM[f"PKG {d}"], color=C1, lw=2, label="PKG")
    for i, c in zip(range(len(SL)), [C2, C3, C4, "#7c5cd6", "#d64d8a", "#5c8a2a"]):
        ax.plot(pd.to_datetime(SHM.month), SHM[f"cm{i} {d}"], color=c, lw=1.5, label=f"check_master: {SL[i][0]}")
    ax.axhline(1, color=INK2, lw=0.8, ls="--")
    ax.set_title(f"Checks {d}: share of the channel view")
    ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
axs[0].legend(fontsize=7, loc="lower left")
savefig(fig, "f01_share_by_month", "PKG and check_master slices as a share of the channel view, by month")
_sc = ["dollar_share_of_view", "item_share_of_view"]
CELL.show(("Whole window: all customer-months, and customers in the view (shares of the view)", pct_tbl(TOT, _sc)),
          ("By month, share of the view (cm = check_master slice number above)", pct_tbl(SHM, [c for c in SHM.columns if c != "month"])))
for d in ("out", "in"):
    t = TOT[TOT.direction == d].set_index("source")
    ans(f"Checks {d}: share of the view, PKG / check_master Debit rule (dollars; items)",
        f"{_pf(t.dollar_share_of_view.get('PKG'))} / {_pf(t.dollar_share_of_view.get('check_master: ' + SL[1][0]))}; "
        f"{_pf(t.item_share_of_view.get('PKG'))} / {_pf(t.item_share_of_view.get('check_master: ' + SL[1][0]))}")

## §3 · Which pieces match

Customer-month by customer-month: share matching within 1% and 5%, the dollar ratio, and the share of customer-months
where the item counts are equal. Pairs: view against check_master slices and PKG; PKG against check_master slices.
For checks in, every combination of the view's deposit columns is tried (lockbox, remote, branch). View pairs use
customer-months in the view; PKG–check_master pairs use every customer-month where either side has checks.

In [ ]:
CELL.start("§3 · Which pieces match")


def _cmp(pre, a, b, an=None, bn=None):
    ok = (a > 0) | (b > 0)
    aggs = [F.sum(F.when(ok, 1).otherwise(0)).alias(f"{pre}_n"),
            F.sum(F.when(ok & (F.abs(a - b) <= CONFIG["TOL"][0] * F.greatest(a, b)), 1).otherwise(0)).alias(f"{pre}_m1"),
            F.sum(F.when(ok & (F.abs(a - b) <= CONFIG["TOL"][1] * F.greatest(a, b)), 1).otherwise(0)).alias(f"{pre}_m5"),
            F.sum(F.when(ok & (b < a * (1 - CONFIG["TOL"][1])), 1).otherwise(0)).alias(f"{pre}_lo"),
            F.sum(F.when(ok & (b <= 0), 1).otherwise(0)).alias(f"{pre}_none"), F.sum(a).alias(f"{pre}_sa"), F.sum(b).alias(f"{pre}_sb")]
    if an is not None:
        aggs.append(F.sum(F.when(ok & (an == bn), 1).otherwise(0)).alias(f"{pre}_ceq"))
    return aggs


with step("pairs"):
    PAIRS = []
    VI = J.where(F.col("in_view") == 1)
    vin_cols = [c for c in CONFIG["VIEW_IN"] if c in CHT]
    vin_combos = [list(c) for n in range(1, len(vin_cols) + 1) for c in itertools.combinations(vin_cols, n)]
    for d in ("out", "in"):
        vcombos = [[c for c in CONFIG["VIEW_OUT"] if c in CHT]] if d == "out" else vin_combos
        targets = [("PKG (all)", F.col(f"pkg_{d}_a"), F.col(f"pkg_{d}_n")), ("PKG, other side outside PNC", F.col(f"pkgx_{d}_a"), F.col(f"pkgx_{d}_n"))] + \
                  [(f"check_master: {lab}", F.col(f"cm{i}_{d}_a"), F.col(f"cm{i}_{d}_n")) for i, (lab, _) in enumerate(SL)]
        specs = []
        for vc in vcombos:
            va = sum((F.col(c) for c in vc), F.lit(0.0))
            vn = sum((F.col(UNIT_OF[c]) for c in vc), F.lit(0.0)) if all(c in UNIT_OF for c in vc) else None
            for lab, b, bn in targets:
                specs.append(("view", " + ".join(c.replace("_amt", "") for c in vc), lab, va, b, vn, bn if vn is not None else None))
        aggs = []
        for k, (_, _, _, a, b, an, bn) in enumerate(specs):
            aggs += _cmp(f"s{k}", a, b, an, bn)
        rr = VI.agg(*aggs).toPandas().pipe(_nodec).iloc[0]
        for k, (src, a_lab, b_lab, *_r) in enumerate(specs):
            n = rr[f"s{k}_n"] or 0
            PAIRS.append(dict(direction=d, left=f"view: {a_lab}", right=b_lab, customer_months=int(n),
                              match_1pct=(rr[f"s{k}_m1"] or 0) / max(n, 1), match_5pct=(rr[f"s{k}_m5"] or 0) / max(n, 1),
                              right_lower=(rr[f"s{k}_lo"] or 0) / max(n, 1), right_none=(rr[f"s{k}_none"] or 0) / max(n, 1),
                              right_over_left=(rr[f"s{k}_sb"] or 0) / (rr[f"s{k}_sa"] or np.nan),
                              counts_equal=(rr[f"s{k}_ceq"] / max(n, 1)) if f"s{k}_ceq" in rr.index else np.nan))
        # PKG against check_master
        aggs, specs2 = [], []
        for i, (lab, _) in enumerate(SL):
            for plab, pa, pn in [("PKG (all)", F.col(f"pkg_{d}_a"), F.col(f"pkg_{d}_n")),
                                 ("PKG, other side outside PNC", F.col(f"pkgx_{d}_a"), F.col(f"pkgx_{d}_n"))]:
                specs2.append((plab, f"check_master: {lab}"))
                k = len(specs2) - 1
                aggs += _cmp(f"t{k}", F.col(f"cm{i}_{d}_a"), pa, F.col(f"cm{i}_{d}_n"), pn)
        rr = J.agg(*aggs).toPandas().pipe(_nodec).iloc[0]
        for k, (plab, clab) in enumerate(specs2):
            n = rr[f"t{k}_n"] or 0
            PAIRS.append(dict(direction=d, left=clab, right=plab, customer_months=int(n),
                              match_1pct=(rr[f"t{k}_m1"] or 0) / max(n, 1), match_5pct=(rr[f"t{k}_m5"] or 0) / max(n, 1),
                              right_lower=(rr[f"t{k}_lo"] or 0) / max(n, 1), right_none=(rr[f"t{k}_none"] or 0) / max(n, 1),
                              right_over_left=(rr[f"t{k}_sb"] or 0) / (rr[f"t{k}_sa"] or np.nan),
                              counts_equal=(rr[f"t{k}_ceq"] or 0) / max(n, 1)))
PAIRS = pd.DataFrame(PAIRS)
PAIRS.to_csv(os.path.join(OUT_DIR, "pairs.csv"), index=False)
_pc = ["match_1pct", "match_5pct", "right_lower", "right_none", "right_over_left", "counts_equal"]
_vp = PAIRS[PAIRS.left.str.startswith("view")].sort_values(["direction", "match_5pct"], ascending=[False, False])
BESTV = _vp.groupby("direction", sort=False).head(1)
CELL.show(("View against PKG and check_master (customer-months in the view; right = PKG or check_master)",
           pct_tbl(_vp.groupby(["direction", "left"], sort=False).head(4), _pc)),
          ("check_master against PKG (left = check_master slice, right = PKG; all customer-months)",
           pct_tbl(PAIRS[~PAIRS.left.str.startswith("view")], _pc)))
for _, r in BESTV.iterrows():
    ans(f"Checks {r.direction}: best match with the view", f"{r.left} = {r.right}: {r.match_1pct:.0%} within 1%, {r.match_5pct:.0%} within 5%, "
                                                           f"ratio {_pf(r.right_over_left)}")
for d in ("out", "in"):
    p = PAIRS[(PAIRS.direction == d) & (PAIRS.left.str.startswith("view")) & (PAIRS.right == "PKG (all)")].sort_values("match_5pct", ascending=False)
    c = PAIRS[(PAIRS.direction == d) & (PAIRS.left.str.startswith("view")) & (PAIRS.right.str.startswith("check_master"))].sort_values("match_5pct", ascending=False)
    if len(p) and len(c):
        ans(f"Checks {d}: match within 5% with the view, PKG / best check_master slice",
            f"{p.iloc[0].match_5pct:.0%} / {c.iloc[0].match_5pct:.0%} ({c.iloc[0].right})")

## §4 · Item by item

For the last `LINK_MONTHS` months, PKG's check payments (category 7 in `neo4j_payments`) are joined to check_master
by id (`trans_id` = `tran_id`; second key: date + the number at the end of the id, in case of zero padding). Then:
which kinds of check_master rows are in PKG, which PKG checks are not in check_master, and do amount and date agree
on the items both hold?

In [ ]:
CELL.start("§4 · Item by item")
_lmo = MONTHS[-CONFIG["LINK_MONTHS"]:] if CONFIG["LINK_MONTHS"] else []
LINK = REV = FMT = AGREE = pd.DataFrame()
if _lmo:
    with step("join PKG check payments to check_master by id"):
        _lo, _hi = f"{_lmo[0]}-01", f"{_lmo[-1]}-31"
        _k2 = lambda idc, dc: F.concat_ws("|", F.substring(F.col(dc).cast("string"), 1, 10),
                                          F.regexp_extract(F.col(idc).cast("string"), r"(\d+)\s*$", 1).cast("decimal(38,0)").cast("string"))
        pay = (spark.table(CONFIG["PAY_TABLE"]).where((F.col("trans_dt") >= _lo) & (F.col("trans_dt") <= _hi))
               .where(F.col("category").cast("string").startswith("7"))
               .select(F.col("trans_id").cast("string").alias("k1"), _k2("trans_id", "trans_dt").alias("k2"),
                       F.upper(F.regexp_extract(F.col("category").cast("string"), r"^\s*(\d[A-Ca-c]\d?)", 1)).alias("code"),
                       F.abs(F.col("trans_amt").cast("double")).alias("p_amt"), F.substring(F.col("trans_dt").cast("string"), 1, 10).alias("p_dt")))
        pay = pay.persist()
        cmr = (cm.where(_dr(_lo, _hi))
               .select(F.col(CONFIG["CM_ID"]).cast("string").alias("k1"), _k2(CONFIG["CM_ID"], CONFIG["CM_DATE"]).alias("k2"),
                       SIDE.alias("side"), *_A, F.abs(F.col(CONFIG["CM_AMT"]).cast("double")).alias("c_amt"), _date.alias("c_dt")))
        p1 = pay.select("k1", F.col("code").alias("code1"), F.col("p_amt").alias("p_amt1"), F.col("p_dt").alias("p_dt1")).dropDuplicates(["k1"])
        p2 = pay.select("k2", F.col("code").alias("code2")).dropDuplicates(["k2"])
        lk = cmr.join(p1, "k1", "left").join(p2, "k2", "left").persist()
        LINK = (lk.groupBy("side", *ATTRS).agg(F.count(F.lit(1)).alias("rows"), F.sum("c_amt").alias("dollars"),
                                               F.sum(F.when(F.col("code1").isNotNull(), F.col("c_amt")).otherwise(0.0)).alias("dollars_in_pkg"),
                                               F.sum(F.when(F.col("code1").isNotNull(), 1).otherwise(0)).alias("rows_in_pkg"),
                                               F.sum(F.when(F.col("code2").isNotNull(), 1).otherwise(0)).alias("rows_in_pkg_key2"),
                                               F.concat_ws(",", F.array_sort(F.collect_set("code1"))).alias("pkg_categories"))
                .toPandas().pipe(_nodec).sort_values("dollars", ascending=False))
        LINK["share_rows_in_pkg"] = LINK.rows_in_pkg / LINK.rows
        LINK["share_dollars_in_pkg"] = LINK.dollars_in_pkg / LINK.dollars
        LINK["share_rows_in_pkg_key2"] = LINK.rows_in_pkg_key2 / LINK.rows
        AGREE = (lk.where(F.col("code1").isNotNull())
                 .agg(F.count(F.lit(1)).alias("linked_rows"),
                      F.avg(F.when(F.abs(F.col("c_amt") - F.col("p_amt1")) <= 0.01, 1).otherwise(0)).alias("same_amount"),
                      F.avg(F.when(F.col("c_dt") == F.col("p_dt1"), 1).otherwise(0)).alias("same_date"),
                      F.avg(F.when(F.datediff(F.col("p_dt1"), F.col("c_dt")) == 1, 1).otherwise(0)).alias("pkg_one_day_later"))
                 .toPandas().pipe(_nodec))
        c1 = cmr.select("k1").distinct().withColumn("in1", F.lit(1))
        c2 = cmr.select("k2").distinct().withColumn("in2", F.lit(1))
        REV = (pay.join(c1, "k1", "left").join(c2, "k2", "left").groupBy("code")
               .agg(F.count(F.lit(1)).alias("pkg_items"), F.sum("p_amt").alias("pkg_dollars"),
                    F.avg(F.coalesce(F.col("in1"), F.lit(0))).alias("share_in_check_master"),
                    F.avg(F.coalesce(F.col("in2"), F.lit(0))).alias("share_in_check_master_key2"))
               .orderBy("code").toPandas().pipe(_nodec))
        _pat = lambda c: F.regexp_replace(F.regexp_replace(F.col(c), r"[0-9]", "9"), r"[A-Za-z]", "A")
        FMT = pd.concat([pay.groupBy(_pat("k1").alias("id_pattern")).count().orderBy(F.desc("count")).limit(5).toPandas().assign(source="PKG"),
                         cmr.groupBy(_pat("k1").alias("id_pattern")).count().orderBy(F.desc("count")).limit(5).toPandas().assign(source="check_master")])
        pay.unpersist()
    LINK.to_csv(os.path.join(OUT_DIR, "link_by_kind.csv"), index=False)
    REV.to_csv(os.path.join(OUT_DIR, "pkg_in_check_master.csv"), index=False)
    _ls = ["share_rows_in_pkg", "share_dollars_in_pkg", "share_rows_in_pkg_key2"]
    CELL.show((f"Id patterns ({_lmo[0]} .. {_lmo[-1]})", FMT[["source", "id_pattern", "count"]]),
              ("PKG check payments found in check_master, by PKG category", pct_tbl(REV, ["share_in_check_master", "share_in_check_master_key2"])),
              (f"check_master rows found in PKG, by kind (top {CONFIG['PROFILE_TOP']}; all in link_by_kind.csv)",
               pct_tbl(LINK.head(CONFIG["PROFILE_TOP"]), _ls)),
              ("Items in both: amount and date agree", pct_tbl(AGREE, ["same_amount", "same_date", "pkg_one_day_later"])))
    _all = LINK[["rows", "rows_in_pkg", "dollars", "dollars_in_pkg"]].sum()
    ans(f"check_master rows found in PKG by id ({_lmo[0]} .. {_lmo[-1]}): rows / dollars",
        f"{_all.rows_in_pkg / max(_all.rows, 1):.1%} / {_all.dollars_in_pkg / max(_all.dollars, 1):.1%}")
    for _, r in REV.iterrows():
        ans(f"PKG {r.code} items found in check_master (id / key 2)", f"{r.share_in_check_master:.1%} / {r.share_in_check_master_key2:.1%}")
else:
    CELL.show(("Item link", pd.DataFrame([dict(status="skipped (LINK_MONTHS = 0)")])))

## §5 · Where PKG falls short

Customer-months in the view with checks, grouped by how PKG compares with the view (none, lower, within 5%, higher).
In each group: what the view, PKG and each check_master slice hold. If check_master fills the gap, its share of the
view in the "PKG none" and "PKG lower" groups is near 100%.

In [ ]:
CELL.start("§5 · Where PKG falls short")
GAP = []
with step("groups by PKG coverage"):
    for d, v in (("out", VOUT), ("in", VIN)):
        p = F.col(f"pkg_{d}_a")
        cls = (F.when(p <= 0, "PKG none").when(F.abs(p - v) <= 0.05 * F.greatest(p, v), "PKG within 5%")
               .when(p < v, "PKG lower").otherwise("PKG higher"))
        g = (VI.where(v > 0).withColumn("cls", cls).groupBy("cls")
             .agg(F.count(F.lit(1)).alias("customer_months"), F.sum(v).alias("view"), F.sum(p).alias("pkg"),
                  *[F.sum(f"cm{i}_{d}_a").alias(f"cm{i}") for i in range(len(SL))])
             .toPandas().pipe(_nodec))
        g["direction"] = d
        g["view_share"] = g.view / g.view.sum()
        g["pkg_share_of_view"] = g.pkg / g.view
        for i in range(len(SL)):
            g[f"cm{i}_share_of_view"] = g[f"cm{i}"] / g.view
        GAP.append(g)
GAP = pd.concat(GAP, ignore_index=True)
_order = {"PKG none": 0, "PKG lower": 1, "PKG within 5%": 2, "PKG higher": 3}
GAP = GAP.assign(_o=GAP.cls.map(_order)).sort_values(["direction", "_o"], ascending=[False, True]).drop(columns="_o")
GAP.to_csv(os.path.join(OUT_DIR, "gap_groups.csv"), index=False)
_gc = ["direction", "cls", "customer_months", "view_share", "pkg_share_of_view"] + [f"cm{i}_share_of_view" for i in range(len(SL))]
CELL.show(("Customer-months in the view by PKG coverage (cm = check_master slice number in §1)",
           pct_tbl(GAP[_gc], ["view_share", "pkg_share_of_view"] + [f"cm{i}_share_of_view" for i in range(len(SL))])))
for d in ("out", "in"):
    g = GAP[(GAP.direction == d) & (GAP.cls.isin(["PKG none", "PKG lower"]))]
    if len(g):
        ans(f"Checks {d}: where PKG has none or less, check_master Debit rule covers (share of view dollars)",
            _pf(g["cm1"].sum() / g.view.sum()) if "cm1" in g else "n/a")

## §6 · Answer sheet

In [ ]:
CELL.start("§6 · Answer sheet")
pd.DataFrame(ANSWERS).to_csv(os.path.join(OUT_DIR, "answers.csv"), index=False)
CELL.show(("Answers", pd.DataFrame(ANSWERS)), ("Gates", gates_frame()),
          (f"Files in {OUT_DIR}", pd.DataFrame(FIG_INDEX, columns=["file", "caption"])),
          ("Timings (minutes)", pd.DataFrame(TIMINGS, columns=["step", "minutes"])))